# Entrenamiento de modelos

Modelos sencillos para las estaciones que tienen histórico y dato actual en el scraper. Cada artefacto se guarda en `models/builded/` como JSON.


In [1]:
from __future__ import annotations

import json
import math
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "time").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "time"
STATIONS_PATH = ROOT / "data" / "estaciones_valencia.csv"
BUILD_DIR = ROOT / "models" / "builded"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

POLLUTANTS = ["SO2", "NO2", "O3", "PM10", "PM2.5"]
TARGET_WINDOWS = {"SO2": 1, "NO2": 1, "O3": 8, "PM10": 24, "PM2.5": 24}
HORIZONS = [8, 24]
LAGS = [8, 24, 48, 168]

RIDGE_ALPHA = 5.0
TRAIN_WINDOW_DAYS = 365 * 3
TEST_WINDOW_DAYS = 31
MIN_TRAIN_ROWS = 500
MIN_TEST_ROWS = 100

print(f"Datos: {DATA_DIR}")
print(f"Salida: {BUILD_DIR}")


Datos: /Users/pablogandia/Desktop/ASIGNATURAS/TERCERO/EDM/proy/data/time
Salida: /Users/pablogandia/Desktop/ASIGNATURAS/TERCERO/EDM/proy/models/builded


In [3]:
def normalize_text(value: str) -> str:
    value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    value = value.lower().replace("avda", "av")
    return re.sub(r"[^a-z0-9]+", " ", value).strip()


def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", normalize_text(value)).strip("_")


def station_from_file(path: Path) -> str:
    return unicodedata.normalize("NFC", path.stem.replace("filtrado_", ""))


def numeric_column(df: pd.DataFrame, name: str) -> pd.Series | None:
    matches = [col for col in df.columns if str(col).strip() == name]
    if not matches:
        return None
    return pd.to_numeric(df[matches[0]], errors="coerce")


In [4]:
def load_history(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, encoding="utf-8-sig")
    hours = pd.to_numeric(raw["HORA"], errors="coerce")
    timestamps = pd.to_datetime(raw["FECHA"], errors="coerce") + pd.to_timedelta(hours, unit="h")

    data = pd.DataFrame(index=timestamps)
    data.index.name = "timestamp"

    for pollutant in POLLUTANTS:
        values = numeric_column(raw, pollutant)
        if values is not None:
            data[pollutant] = values.to_numpy()

    data = data[~data.index.isna()]
    data = data[~data.index.duplicated(keep="first")].sort_index()
    return data.asfreq("h")


def scraper_targets(hourly: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=hourly.index)
    for pollutant in POLLUTANTS:
        if pollutant not in hourly:
            continue
        window = TARGET_WINDOWS[pollutant]
        min_periods = 1 if window == 1 else math.ceil(window * 0.75)
        out[pollutant] = hourly[pollutant].rolling(window, min_periods=min_periods).mean()
    return out


In [5]:
def make_features(targets: pd.DataFrame, pollutant: str, horizon: int) -> tuple[pd.DataFrame, pd.Series]:
    X = pd.DataFrame(index=targets.index)

    for col in targets.columns:
        X[f"{col}_current"] = targets[col]

    base = targets[pollutant]
    for lag in LAGS:
        X[f"{pollutant}_lag_{lag}h"] = base.shift(lag)

    X[f"{pollutant}_rolling_24h"] = base.rolling(24, min_periods=18).mean()
    X[f"{pollutant}_rolling_7d"] = base.rolling(168, min_periods=126).mean()

    hour = X.index.hour
    dow = X.index.dayofweek
    month = X.index.month
    X["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    X["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    X["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    X["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    X["month_sin"] = np.sin(2 * np.pi * month / 12)
    X["month_cos"] = np.cos(2 * np.pi * month / 12)

    y = base.shift(-horizon).rename("target")
    dataset = pd.concat([X, y], axis=1).dropna()
    return dataset.drop(columns="target"), dataset["target"]


In [6]:
def fit_ridge(X: pd.DataFrame, y: pd.Series, alpha: float) -> dict:
    mean = X.mean()
    std = X.std().replace(0, 1)
    Xs = ((X - mean) / std).to_numpy(dtype=float)
    yd = y.to_numpy(dtype=float)

    design = np.column_stack([np.ones(len(Xs)), Xs])
    penalty = np.eye(design.shape[1]) * alpha
    penalty[0, 0] = 0
    beta = np.linalg.solve(design.T @ design + penalty, design.T @ yd)

    return {
        "intercept": float(beta[0]),
        "coefficients": [float(v) for v in beta[1:]],
        "feature_means": {k: float(v) for k, v in mean.items()},
        "feature_stds": {k: float(v) for k, v in std.items()},
    }


def predict(model: dict, X: pd.DataFrame) -> np.ndarray:
    cols = model["feature_names"]
    mean = pd.Series(model["feature_means"])[cols]
    std = pd.Series(model["feature_stds"])[cols]
    Xs = ((X[cols] - mean) / std).to_numpy(dtype=float)
    return model["intercept"] + Xs @ np.array(model["coefficients"])


def metrics(y: pd.Series, pred: np.ndarray) -> dict:
    actual = y.to_numpy(dtype=float)
    mae = float(np.mean(np.abs(actual - pred)))
    rmse = float(np.sqrt(np.mean((actual - pred) ** 2)))
    denom = float(np.sum((actual - actual.mean()) ** 2))
    r2 = float("nan") if denom == 0 else float(1 - np.sum((actual - pred) ** 2) / denom)
    return {"mae": mae, "rmse": rmse, "r2": r2}


In [7]:
def train_one(station: str, source_file: Path, targets: pd.DataFrame, pollutant: str, horizon: int, trained_at: str) -> dict | None:
    X, y = make_features(targets, pollutant, horizon)
    if X.empty:
        return None

    last_ts = targets.index.max()
    train_min = last_ts - pd.Timedelta(days=TRAIN_WINDOW_DAYS)
    test_min = last_ts - pd.Timedelta(days=TEST_WINDOW_DAYS)

    train_mask = (X.index >= train_min) & (X.index < test_min)
    test_mask = X.index >= test_min
    X_train, y_train = X.loc[train_mask], y.loc[train_mask]
    X_test, y_test = X.loc[test_mask], y.loc[test_mask]

    if len(X_train) < MIN_TRAIN_ROWS or len(X_test) < MIN_TEST_ROWS:
        return None

    model = fit_ridge(X_train, y_train, RIDGE_ALPHA)
    model.update({
        "model_type": "ridge_numpy",
        "station": station,
        "pollutant": pollutant,
        "horizon_hours": horizon,
        "target_window_hours": TARGET_WINDOWS[pollutant],
        "alpha": RIDGE_ALPHA,
        "feature_names": list(X_train.columns),
        "trained_at_utc": trained_at,
        "source_file": str(source_file.relative_to(ROOT)),
        "source_start": str(targets.index.min()),
        "source_end": str(targets.index.max()),
        "train_start": str(X_train.index.min()),
        "train_end": str(X_train.index.max()),
        "test_start": str(X_test.index.min()),
        "test_end": str(X_test.index.max()),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
    })

    pred = predict(model, X_test)
    baseline = X_test[f"{pollutant}_current"].to_numpy(dtype=float)
    model["metrics"] = metrics(y_test, pred)
    model["baseline_metrics"] = metrics(y_test, baseline)
    model["beats_baseline_mae"] = bool(model["metrics"]["mae"] <= model["baseline_metrics"]["mae"])
    return model


In [8]:
for old_file in BUILD_DIR.glob("model__*.json"):
    old_file.unlink()

trained_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
registry = []
metric_rows = []

for csv_path in sorted(DATA_DIR.glob("*.csv")):
    station = station_from_file(csv_path)
    targets = scraper_targets(load_history(csv_path))

    for pollutant in POLLUTANTS:
        if pollutant not in targets or targets[pollutant].dropna().empty:
            continue

        for horizon in HORIZONS:
            model = train_one(station, csv_path, targets, pollutant, horizon, trained_at)
            if model is None:
                continue

            filename = f"model__{slugify(station)}__{slugify(pollutant)}__h{horizon:02d}.json"
            model_path = BUILD_DIR / filename
            model_path.write_text(json.dumps(model, indent=2, ensure_ascii=False), encoding="utf-8")

            row = {
                "trained_at_utc": trained_at,
                "station": station,
                "pollutant": pollutant,
                "horizon_hours": horizon,
                "mae": model["metrics"]["mae"],
                "rmse": model["metrics"]["rmse"],
                "r2": model["metrics"]["r2"],
                "baseline_mae": model["baseline_metrics"]["mae"],
                "baseline_rmse": model["baseline_metrics"]["rmse"],
                "baseline_r2": model["baseline_metrics"]["r2"],
                "beats_baseline_mae": model["beats_baseline_mae"],
                "train_rows": model["train_rows"],
                "test_rows": model["test_rows"],
                "model_path": str(model_path.relative_to(ROOT)),
            }
            metric_rows.append(row)
            registry.append({
                "station": station,
                "pollutant": pollutant,
                "horizon_hours": horizon,
                "model_path": row["model_path"],
                "mae": row["mae"],
                "baseline_mae": row["baseline_mae"],
                "beats_baseline_mae": row["beats_baseline_mae"],
                "trained_at_utc": trained_at,
            })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(BUILD_DIR / "metrics.csv", index=False, encoding="utf-8-sig")
(BUILD_DIR / "registry.json").write_text(json.dumps(registry, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Modelos entrenados: {len(registry)}")
print(f"Artefactos: {BUILD_DIR.relative_to(ROOT)}")


Modelos entrenados: 64
Artefactos: models/builded


In [9]:
summary = (
    metrics_df
    .groupby(["pollutant", "horizon_hours"], as_index=False)
    .agg(
        models=("model_path", "count"),
        mae=("mae", "mean"),
        baseline_mae=("baseline_mae", "mean"),
        beats_baseline=("beats_baseline_mae", "sum"),
    )
)
summary["mae_improvement_pct"] = 100 * (summary["baseline_mae"] - summary["mae"]) / summary["baseline_mae"]

print(f"Mejoran baseline: {int(metrics_df['beats_baseline_mae'].sum())}/{len(metrics_df)}")
display(summary.sort_values(["pollutant", "horizon_hours"]))

display(
    metrics_df
    .assign(mae_delta=lambda d: d["baseline_mae"] - d["mae"])
    .sort_values("mae_delta", ascending=False)
    .head(10)[["station", "pollutant", "horizon_hours", "mae", "baseline_mae", "mae_delta", "r2"]]
)


Mejoran baseline: 47/64


,pollutant,horizon_hours,models,mae,baseline_mae,beats_baseline,mae_improvement_pct
0,NO2,8,8,10.634991,14.702779,8,27.666798
1,NO2,24,8,10.255391,10.300460,5,0.437545
2,O3,8,6,10.338276,10.794038,3,4.222344
3,O3,24,6,11.679987,12.501076,6,6.568148
4,PM10,8,6,2.272754,2.869419,6,20.793929
5,PM10,24,6,6.100654,6.847627,5,10.908494
6,PM2.5,8,6,1.131395,1.521282,6,25.628868
7,PM2.5,24,6,3.318490,3.717356,6,10.729851
8,SO2,8,6,0.430814,0.401286,2,-7.358385
9,SO2,24,6,0.442223,0.315816,0,-40.025593


,station,pollutant,horizon_hours,mae,baseline_mae,mae_delta,r2
58,València Olivereta,NO2,8,13.312805,21.615721,8.302915,0.349893
16,València - Centre,NO2,8,10.852006,15.534985,4.682979,0.266407
54,València - Vivers,NO2,8,11.538224,16.119670,4.581446,0.199427
34,València - Pista de Silla,NO2,8,11.862521,16.284712,4.422191,0.253125
2,València - Av. França,NO2,8,11.098789,15.437037,4.338248,0.357740
24,València - Molí del Sol,NO2,8,9.063728,12.112299,3.048572,0.196423
61,València Olivereta,PM10,24,6.728760,9.018889,2.290128,0.437867
19,València - Centre,PM10,24,7.953450,9.700768,1.747319,0.317955
12,València - Bulevard Sud,NO2,8,8.065831,9.658407,1.592576,0.141629
44,València - Politècnic,NO2,8,9.286020,10.859397,1.573377,0.197240
